In [2]:
import sys
!{sys.executable} -m pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 27.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [3]:
import duckdb
import pandas as pd
import numpy as np

In [5]:
con = duckdb.connect("../data/trading.duckdb")
df = con.execute("SELECT * FROM daily_duol ORDER BY timestamp").fetchdf()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')

# 1. Volatility — tells you how reactive your MAs should be
vol = df['close'].pct_change().std() * np.sqrt(252)
print(f"Annualized vol: {vol:.1%}")  # if >60%, use shorter MAs

# 2. Backtest common MA pairs
def backtest(df, short_w, long_w):
    df = df.copy()
    df['sma_short'] = df['close'].rolling(short_w).mean()
    df['sma_long']  = df['close'].rolling(long_w).mean()
    df = df.dropna()
    df['signal']   = np.where(df['sma_short'] > df['sma_long'], 1, -1)
    df['position'] = df['signal'].shift(1)
    df['ret']      = df['close'].pct_change()
    df['strat']    = df['position'] * df['ret']
    sharpe = df['strat'].mean() / df['strat'].std() * np.sqrt(252)
    total  = (1 + df['strat']).prod() - 1
    trades = int(df['signal'].diff().abs().sum() / 2)
    print(f"  SMA {short_w}/{long_w}: Sharpe={sharpe:.2f}, Return={total:.1%}, Trades={trades}")

for s, l in [(10,50), (20,50), (20,100), (50,200)]:
    backtest(df, s, l)



Annualized vol: 67.0%
  SMA 10/50: Sharpe=-0.35, Return=-86.7%, Trades=33
  SMA 20/50: Sharpe=-0.09, Return=-71.4%, Trades=27
  SMA 20/100: Sharpe=0.09, Return=-49.6%, Trades=12
  SMA 50/200: Sharpe=0.20, Return=-26.3%, Trades=6
